# 🧠 EMIPredict AI: Model Training Masterclass

This notebook documents the training and evaluation of our dual-model system:
1. **Classification**: To predict Loan Eligibility (Approved / High Risk / Not Eligible).
2. **Regression**: To predict the Maximum Safe EMI for an applicant.

---

## 🛠️ Step 1: Environment Setup
We use **MLflow** for experiment tracking. This allows us to compare different algorithms (Logistic Regression vs. XGBoost) and save the best ones.

In [1]:
import mlflow
import mlflow.sklearn
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import sys
import os

# Set up paths to import our custom preprocessing logic
module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.append(module_path)

from preprocessing import load_and_engineer

project_root = Path('..')
mlflow.set_tracking_uri(f"file://{project_root.absolute()}/mlruns")

## 🧱 Step 2: Load Preprocessed Data
We use our centralized `preprocessing.py` to get the training/testing sets. This ensures 100% consistency.

In [2]:
X_train, X_test, yc_train, yc_test, yr_train, yr_test, feat_cols = load_and_engineer()
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Features after engineering: 33 columns
Target classes: ['Eligible' 'High_Risk' 'Not_Eligible']
Scaler and encoders saved to /home/ganesaperumal/Documents/DS/claude/EMIPredict_AI/models/

Train size: (323840, 31), Test size: (80960, 31)
Training set shape: (323840, 31)
Testing set shape: (80960, 31)


## 🧪 Step 3: Classification Training (Eligibility)
We compare three algorithms: **Logistic Regression**, **Random Forest**, and **XGBoost**.

**Why XGBoost?** It is an ensemble method that uses Gradient Boosting. It is highly efficient and resistant to overfitting.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_experiment("EMI_Classification")

def train_clf(model, name):
    with mlflow.start_run(run_name=name):
        model.fit(X_train, yc_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(yc_test, y_pred)
        f1  = f1_score(yc_test, y_pred, average='weighted')
        
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        
        print(f"{name} -> Accuracy: {acc:.4f}, F1: {f1:.4f}")
        return acc, f1, model

clf_models = [
    (LogisticRegression(max_iter=1000, random_state=42), "Logistic_Regression"),
    (RandomForestClassifier(n_estimators=100, random_state=42), "Random_Forest"),
    (XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss'), "XGBoost")
]

clf_results = []
for model, name in clf_models:
    acc, f1, trained = train_clf(model, name)
    clf_results.append((name, acc, f1, trained))

/home/ganesaperumal/Documents/DS/claude/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Logistic_Regression -> Accuracy: 0.9120, F1: 0.8927
Random_Forest -> Accuracy: 0.9506, F1: 0.9368
XGBoost -> Accuracy: 0.9747, F1: 0.9730


## 💰 Step 4: Regression Training (Max EMI)
Now we train a model to predict the continuous value of **Max EMI allowed**.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

mlflow.set_experiment("EMI_Regression")

def train_reg(model, name):
    with mlflow.start_run(run_name=name):
        model.fit(X_train, yr_train)
        y_pred = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(yr_test, y_pred))
        r2   = r2_score(yr_test, y_pred)
        
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        
        print(f"{name} -> RMSE: {rmse:.2f}, R2: {r2:.4f}")
        return rmse, r2, model

reg_models = [
    (LinearRegression(), "Linear_Regression"),
    (RandomForestRegressor(n_estimators=100, random_state=42), "Random_Forest"),
    (XGBRegressor(n_estimators=100, random_state=42), "XGBoost")
]

reg_results = []
for model, name in reg_models:
    rmse, r2, trained = train_reg(model, name)
    reg_results.append((name, rmse, r2, trained))

Linear_Regression -> RMSE: 4137.59, R2: 0.7166


## 🏆 Step 5: Final Selection & Persistence
We select the best performing models and save them to the `models/` directory using **Pickle**.

In [4]:
# Best Classifier by F1-Score
best_clf = max(clf_results, key=lambda x: x[2])
print(f"🏆 Best Classifier: {best_clf[0]}")

# Best Regressor by RMSE (Lower is better)
best_reg = min(reg_results, key=lambda x: x[1])
print(f"🏆 Best Regressor: {best_reg[0]}")

# Save artifacts
models_path = project_root / 'models'
models_path.mkdir(exist_ok=True)

pickle.dump(best_clf[3], open(models_path / 'best_classifier.pkl', 'wb'))
pickle.dump(best_reg[3], open(models_path / 'best_regressor.pkl', 'wb'))

print("\n✅ Models saved successfully to models/ directory.")

🏆 Best Classifier: XGBoost


NameError: name 'reg_results' is not defined